In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb


In [2]:
df = pd.read_csv('../../data/cleaned_df.csv')
df = df.drop_duplicates()
df = df[df['StateOrProvince'] == 'CA']
df.head()

/var/folders/5g/sd7vmfvs2rn86tg601yfsjx80000gn/T/ipykernel_77128/3836542674.py:1: DtypeWarning: Columns (0: PoolPrivateYN, 1: FireplaceYN) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../../data/cleaned_df.csv')


,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ListingContractDate,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude
0,1800000.0,3546.0,7740.0,4.0,2.0,1.0,5.0,0,NaN,2856 Muir Trail Drive,...,2025-03-24,CA,True,False,True,True,False,3.0,-117.977995,33.899427
1,1425000.0,1672.0,7500.0,2.0,1.0,3.0,3.0,0,NaN,5305 Via Cartagena,...,2025-03-10,CA,True,False,True,True,False,2.0,-117.778002,33.884526
2,875000.0,1111.0,5353.0,2.0,1.0,2.0,2.0,0,NaN,13741 Oak Crest Drive,...,2025-05-30,CA,False,False,True,False,False,2.0,-118.038512,33.878294
3,575000.0,1273.0,6902.0,2.0,1.0,3.0,3.0,0,"Carpet,Vinyl",5630 Kingsley Street,...,2025-04-18,CA,True,False,True,True,False,2.0,-117.682399,34.066983
4,2250000.0,2000.0,21430.0,2.0,1.0,4.0,4.0,0,NaN,128 W Las Flores Avenue,...,2025-04-22,CA,True,True,True,True,False,2.0,-118.035862,34.115673


In [3]:
import geopandas as gpd

districts = gpd.read_file('../../data/CA_district_areas.geojson')
districts = districts.to_crs("EPSG:4326")
districts.head()

,OBJECTID,Year,FedID,CDCode,CDSCode,CountyName,DistrictName,DistrictType,GradeLow,GradeHigh,...,MIGcount,MIGpct,SWDcount,SWDpct,SEDcount,SEDpct,DistrctAreaSqMi,LocaleCode,LocaleDesc,geometry
0,1,2025-26,0601770,0161119,01611190000000,Alameda,Alameda Unified,Unified,PK,12,...,0,0.0,1302,12.1,4259,39.5,11.248886,21,"21 - Suburban, Large","MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,2,2025-26,0601860,0161127,01611270000000,Alameda,Albany City Unified,Unified,PK,12,...,0,0.0,363,9.7,1247,33.3,1.789975,21,"21 - Suburban, Large","POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,3,2025-26,0604740,0161143,01611430000000,Alameda,Berkeley Unified,Unified,PK,12,...,0,0.0,1118,11.9,2710,28.8,10.434281,12,"12 - City, Midsize","POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,4,2025-26,0607800,0161150,01611500000000,Alameda,Castro Valley Unified,Unified,PK,12,...,2,0.0,1186,12.2,3784,39.0,66.885261,21,"21 - Suburban, Large","MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,5,2025-26,0612630,0161168,01611680000000,Alameda,Emery Unified,Unified,PK,12,...,0,0.0,98,16.1,407,66.7,1.273923,21,"21 - Suburban, Large","POLYGON ((-122.29663 37.8311, -122.29778 37.83..."


In [4]:
district_info = districts[['DistrictName', 'DistrictType', 'geometry']].copy()
district_info.head()

,DistrictName,DistrictType,geometry
0,Alameda Unified,Unified,"MULTIPOLYGON (((-122.22678 37.72651, -122.2267..."
1,Albany City Unified,Unified,"POLYGON ((-122.28671 37.89852, -122.28673 37.8..."
2,Berkeley Unified,Unified,"POLYGON ((-122.25606 37.89834, -122.25607 37.8..."
3,Castro Valley Unified,Unified,"MULTIPOLYGON (((-122.01375 37.64265, -122.0114..."
4,Emery Unified,Unified,"POLYGON ((-122.29663 37.8311, -122.29778 37.83..."


In [5]:
geo_df = gpd.GeoDataFrame(df.copy(), geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), crs='EPSG:4326')
geo_df.head()

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude,geometry
0,1800000.0,3546.0,7740.0,4.0,2.0,1.0,5.0,0,NaN,2856 Muir Trail Drive,...,CA,True,False,True,True,False,3.0,-117.977995,33.899427,POINT (-117.978 33.89943)
1,1425000.0,1672.0,7500.0,2.0,1.0,3.0,3.0,0,NaN,5305 Via Cartagena,...,CA,True,False,True,True,False,2.0,-117.778002,33.884526,POINT (-117.778 33.88453)
2,875000.0,1111.0,5353.0,2.0,1.0,2.0,2.0,0,NaN,13741 Oak Crest Drive,...,CA,False,False,True,False,False,2.0,-118.038512,33.878294,POINT (-118.03851 33.87829)
3,575000.0,1273.0,6902.0,2.0,1.0,3.0,3.0,0,"Carpet,Vinyl",5630 Kingsley Street,...,CA,True,False,True,True,False,2.0,-117.682399,34.066983,POINT (-117.6824 34.06698)
4,2250000.0,2000.0,21430.0,2.0,1.0,4.0,4.0,0,NaN,128 W Las Flores Avenue,...,CA,True,True,True,True,False,2.0,-118.035862,34.115673,POINT (-118.03586 34.11567)


In [6]:
df_districts = gpd.sjoin(geo_df, district_info, how="left", predicate="within").drop(columns=['index_right', 'geometry'])
df_districts.head()

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude,DistrictName,DistrictType
0,1800000.0,3546.0,7740.0,4.0,2.0,1.0,5.0,0,NaN,2856 Muir Trail Drive,...,True,False,True,True,False,3.0,-117.977995,33.899427,Buena Park Elementary,Elementary
0,1800000.0,3546.0,7740.0,4.0,2.0,1.0,5.0,0,NaN,2856 Muir Trail Drive,...,True,False,True,True,False,3.0,-117.977995,33.899427,Fullerton Joint Union High,High
1,1425000.0,1672.0,7500.0,2.0,1.0,3.0,3.0,0,NaN,5305 Via Cartagena,...,True,False,True,True,False,2.0,-117.778002,33.884526,Placentia-Yorba Linda Unified,Unified
2,875000.0,1111.0,5353.0,2.0,1.0,2.0,2.0,0,NaN,13741 Oak Crest Drive,...,False,False,True,False,False,2.0,-118.038512,33.878294,ABC Unified,Unified
3,575000.0,1273.0,6902.0,2.0,1.0,3.0,3.0,0,"Carpet,Vinyl",5630 Kingsley Street,...,True,False,True,True,False,2.0,-117.682399,34.066983,Chaffey Joint Union High,High


In [7]:
district_features = (df_districts.reset_index().pivot_table(index='index', columns='DistrictType', values='DistrictName', aggfunc='first'))
district_features

DistrictType,Elementary,High,Unified
index,,,
0,Buena Park Elementary,Fullerton Joint Union High,NaN
1,NaN,NaN,Placentia-Yorba Linda Unified
2,NaN,NaN,ABC Unified
3,Ontario-Montclair,Chaffey Joint Union High,NaN
4,NaN,NaN,Arcadia Unified
...,...,...,...
87598,NaN,NaN,Palm Springs Unified
87599,NaN,NaN,Atascadero Unified
87600,Pioneer Union Elementary,Oroville Union High,NaN


In [8]:
district_split = district_features.rename(columns={'Elementary': 'ElementaryDistrict', 'High': 'HighDistrict', 'Unified': 'UnifiedDistrict'})
district_split

DistrictType,ElementaryDistrict,HighDistrict,UnifiedDistrict
index,,,
0,Buena Park Elementary,Fullerton Joint Union High,NaN
1,NaN,NaN,Placentia-Yorba Linda Unified
2,NaN,NaN,ABC Unified
3,Ontario-Montclair,Chaffey Joint Union High,NaN
4,NaN,NaN,Arcadia Unified
...,...,...,...
87598,NaN,NaN,Palm Springs Unified
87599,NaN,NaN,Atascadero Unified
87600,Pioneer Union Elementary,Oroville Union High,NaN


In [9]:
df = df_districts.join(district_split).drop(columns=['DistrictType', 'DistrictName', 'ElementaryDistrict', 'HighDistrict']).drop_duplicates()
df

,ClosePrice,LivingArea,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,Flooring,UnparsedAddress,...,StateOrProvince,ViewYN,PoolPrivateYN,AttachedGarageYN,FireplaceYN,NewConstructionYN,ParkingTotal,Longitude,Latitude,UnifiedDistrict
0,1800000.0,3546.0,7740.0,4.0,2.0,1.0,5.0,0,NaN,2856 Muir Trail Drive,...,CA,True,False,True,True,False,3.0,-117.977995,33.899427,NaN
1,1425000.0,1672.0,7500.0,2.0,1.0,3.0,3.0,0,NaN,5305 Via Cartagena,...,CA,True,False,True,True,False,2.0,-117.778002,33.884526,Placentia-Yorba Linda Unified
2,875000.0,1111.0,5353.0,2.0,1.0,2.0,2.0,0,NaN,13741 Oak Crest Drive,...,CA,False,False,True,False,False,2.0,-118.038512,33.878294,ABC Unified
3,575000.0,1273.0,6902.0,2.0,1.0,3.0,3.0,0,"Carpet,Vinyl",5630 Kingsley Street,...,CA,True,False,True,True,False,2.0,-117.682399,34.066983,NaN
4,2250000.0,2000.0,21430.0,2.0,1.0,4.0,4.0,0,NaN,128 W Las Flores Avenue,...,CA,True,True,True,True,False,2.0,-118.035862,34.115673,Arcadia Unified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87598,565000.0,1703.0,8276.0,2.0,1.0,4.0,4.0,162,"Carpet,Tile",69722 Ridgeway Avenue,...,CA,True,True,True,True,False,2.0,-116.445631,33.829444,Palm Springs Unified
87599,605000.0,1032.0,6534.0,2.0,1.0,3.0,3.0,100,NaN,8920 Arcade Road,...,CA,True,False,True,False,False,2.0,-120.651240,35.475934,Atascadero Unified
87600,160000.0,1192.0,161172.0,1.0,1.0,1.0,1.0,168,"Carpet,Vinyl",1171 Bald Rock Road,...,CA,True,False,NaN,True,False,0.0,-121.395870,39.637090,NaN
87601,340000.0,2023.0,10454.0,3.0,1.0,1.0,6.0,174,NaN,7871 Fernwood Avenue,...,CA,True,False,True,True,False,2.0,-117.987498,35.120506,Mojave Unified


In [10]:
df['UnifiedDistrict'].isna().sum()/len(df)

np.float64(0.24292237442922374)

In [11]:
model_df = df.copy()
# Building to lot ratio
model_df["LotLivingRatio"] = np.where(df["LotSizeSquareFeet"] > 0, df["LivingArea"] / df["LotSizeSquareFeet"], np.nan)
# Bedroom to bathroom ratio
model_df['BathroomBedroomRatio'] = np.where(df['BedroomsTotal'] > 0, df['BathroomsTotalInteger']/df['BedroomsTotal'], np.nan)
# Total amount of house amenities
amenity_cols = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN']
amenities = df[amenity_cols].fillna(False).astype(int)
model_df['TotalAmenityCount'] = amenities.sum(axis=1)
# Total number of different types of flooring
model_df['FlooringCount'] = df['Flooring'].str.split(',').str.len()

In [12]:
model_df.columns

Index(['ClosePrice', 'LivingArea', 'LotSizeSquareFeet',
       'BathroomsTotalInteger', 'Stories', 'MainLevelBedrooms',
       'BedroomsTotal', 'DaysOnMarket', 'Flooring', 'UnparsedAddress',
       'AssociationFeeFrequency', 'MLSAreaMajor', 'ElementarySchool',
       'SubdivisionName', 'City', 'PurchaseContractDate',
       'MiddleOrJuniorSchool', 'HighSchool', 'HighSchoolDistrict', 'Levels',
       'ListingKey', 'CloseDate', 'PropertyType', 'ListingKeyNumeric',
       'CountyOrParish', 'MlsStatus', 'PropertySubType', 'ListingId',
       'ContractStatusChangeDate', 'ListingContractDate', 'StateOrProvince',
       'ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN',
       'NewConstructionYN', 'ParkingTotal', 'Longitude', 'Latitude',
       'UnifiedDistrict', 'LotLivingRatio', 'BathroomBedroomRatio',
       'TotalAmenityCount', 'FlooringCount'],
      dtype='str')

In [13]:
target = 'ClosePrice'

# Categorize columns

cat_col = ['Flooring', 'AssociationFeeFrequency', 
        'MLSAreaMajor', 'ElementarySchool', 'SubdivisionName', 'City', 
        'PurchaseContractDate', 'MiddleOrJuniorSchool', 'HighSchool',
        'HighSchoolDistrict', 'Levels', 'ListingKey', 'CloseDate', 
        'PropertyType', 'ListingKeyNumeric', 'CountyOrParish',
        'PropertySubType', 'ListingId', 'ContractStatusChangeDate',
        'ListingContractDate', 'StateOrProvince', 'UnifiedDistrict']

bool_col = ['ViewYN', 'PoolPrivateYN', 'AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN']

num_col = ['LivingArea', 'LotSizeSquareFeet', 'BathroomsTotalInteger', 
            'Stories', 'MainLevelBedrooms', 'BedroomsTotal', 'DaysOnMarket',
            'LotLivingRatio', 'BathroomBedroomRatio', 'TotalAmenityCount', 'FlooringCount']

required_cols = ['ParkingTotal', 'Longitude', 'Latitude']


# Keep only columns that exist in the dataframe
num_col = [col for col in num_col if col in model_df.columns]
cat_col = [col for col in cat_col if col in model_df.columns]
bool_col = [col for col in bool_col if col in model_df.columns]

keep_cols = [target] + cat_col + bool_col + required_cols + num_col 
model_df = model_df[keep_cols]
# model_df = model_df.drop_duplicates()

model_df.head()


,ClosePrice,Flooring,AssociationFeeFrequency,MLSAreaMajor,ElementarySchool,SubdivisionName,City,PurchaseContractDate,MiddleOrJuniorSchool,HighSchool,...,LotSizeSquareFeet,BathroomsTotalInteger,Stories,MainLevelBedrooms,BedroomsTotal,DaysOnMarket,LotLivingRatio,BathroomBedroomRatio,TotalAmenityCount,FlooringCount
0,1800000.0,NaN,Monthly,83 - Fullerton,NaN,Cardinal Crest (CARD),Fullerton,2025-03-26,NaN,NaN,...,7740.0,4.0,2.0,1.0,5.0,0,0.458140,0.800000,3,NaN
1,1425000.0,NaN,Monthly,85 - Yorba Linda,NaN,East Lake Village Homes (ELVH),Yorba Linda,2025-03-10,NaN,NaN,...,7500.0,2.0,1.0,3.0,3.0,0,0.222933,0.666667,3,NaN
2,875000.0,NaN,NaN,RA - Cerritos North of 91 Frwy,Gonsalves,NaN,Cerritos,2025-05-30,Carnegie,Cerritos,...,5353.0,2.0,1.0,2.0,2.0,0,0.207547,1.000000,1,NaN
3,575000.0,"Carpet,Vinyl",NaN,685 - Montclair,NaN,NaN,Montclair,2025-04-18,NaN,NaN,...,6902.0,2.0,1.0,3.0,3.0,0,0.184439,0.666667,3,2.0
4,2250000.0,NaN,NaN,605 - Arcadia,NaN,NaN,Arcadia,2025-04-25,NaN,NaN,...,21430.0,2.0,1.0,4.0,4.0,0,0.093327,0.500000,4,NaN


### Model Building

In [14]:

def preproc_df(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()

    # ----------------------------
    # Drop rows with missing values in key columns
    # ----------------------------
    required_cols = ['ParkingTotal', 'Longitude', 'Latitude']
    train_df = train_df.dropna(subset=required_cols)
    test_df = test_df.dropna(subset=required_cols)

    # Remove invalid values
    train_df = train_df[
        (train_df["ClosePrice"] > 0) &
        (train_df["LivingArea"] > 0) &
        (train_df["BathroomsTotalInteger"] > 0) &
        (train_df["DaysOnMarket"] > 0)
    ]

    test_df = test_df[
        (test_df["ClosePrice"] > 0) &
        (test_df["LivingArea"] > 0) &
        (test_df["BathroomsTotalInteger"] > 0) &
        (test_df["DaysOnMarket"] > 0)
    ]

    # ----------------------------
    # Missing value handling
    # ----------------------------

    for col in cat_col:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna("Unknown")
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna("Unknown")

    # Missing indicator creation
    for col in train_df.columns:
        if train_df[col].isna().sum() > 0:
            train_df[f"{col}_was_missing"] = train_df[col].isna().astype(int) #
            test_df[f"{col}_was_missing"] = test_df[col].isna().astype(int) #

    if "YearBuilt" in train_df.columns:
        median = train_df["YearBuilt"].median()
        train_df["YearBuilt"] = train_df["YearBuilt"].fillna(median)
        test_df["YearBuilt"] = test_df["YearBuilt"].fillna(median)

    if "StreetNumberNumeric" in train_df.columns:
        train_df["StreetNumberNumeric"] = train_df["StreetNumberNumeric"].fillna(-1)
        test_df["StreetNumberNumeric"] = test_df["StreetNumberNumeric"].fillna(-1)

    train_df[bool_col] = train_df[bool_col].fillna(False)
    test_df[bool_col] = test_df[bool_col].fillna(False)

    for col in num_col:
        if col in train_df.columns:
            median = train_df[col].median()
            train_df[col] = train_df[col].fillna(median)
            test_df[col] = test_df[col].fillna(median)

    if "AssociationFee" in train_df.columns:
        train_df["AssociationFee"] = train_df["AssociationFee"].fillna(0)
        test_df["AssociationFee"] = test_df["AssociationFee"].fillna(0)

    for col in ["GarageSpaces", "ParkingTotal"]:
        if col in train_df.columns:
            train_df[col] = train_df[col].fillna(0)
            test_df[col] = test_df[col].fillna(0)

    # ----------------------------
    # Boolean encoding
    # ----------------------------

    train_df[bool_col] = train_df[bool_col].astype(int)
    test_df[bool_col] = test_df[bool_col].astype(int)

    # ----------------------------
    # MultiLabel Encoding
    # ----------------------------

    multi_cols = ["Flooring", "Levels"]

    for col in multi_cols:
        train_df[col] = train_df[col].fillna("").str.split(",")
        test_df[col] = test_df[col].fillna("").str.split(",")

        mlb = MultiLabelBinarizer()

        train_encoded = pd.DataFrame(
            mlb.fit_transform(train_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=train_df.index
        )

        test_encoded = pd.DataFrame(
            mlb.transform(test_df[col]),
            columns=[f"{col}_{c}" for c in mlb.classes_],
            index=test_df.index
        )

        train_df = train_df.drop(columns=col).join(train_encoded)
        test_df = test_df.drop(columns=col).join(test_encoded)

    # ----------------------------
    # Ordinal Encoding
    # ----------------------------

    mapping = {
        "Unknown": 0,
        "Monthly": 1,
        "Quarterly": 2,
        "SemiAnnually": 3,
        "Annually": 4
    }

    train_df["AssociationFeeFrequency"] = train_df["AssociationFeeFrequency"].map(mapping)
    test_df["AssociationFeeFrequency"] = test_df["AssociationFeeFrequency"].map(mapping)

    # ----------------------------
    # One-Hot Encoding
    # ----------------------------

    one_hot = [
        "CountyOrParish",
        "StateOrProvince",
        "City",
        "PropertyType",
        "PropertySubType",
        'UnifiedDistrict'
    ]

    for col in one_hot:
        top = train_df[col].value_counts().head(200).index

        train_df[col] = train_df[col].where(train_df[col].isin(top), "Other")
        test_df[col] = test_df[col].where(test_df[col].isin(top), "Other")

    train_df = pd.get_dummies(train_df, columns=one_hot, drop_first=True)
    test_df = pd.get_dummies(test_df, columns=one_hot, drop_first=True)

    # Ensure same columns
    train_df, test_df = train_df.align(test_df, join="left", axis=1, fill_value=0)

    # ----------------------------
    # Standardization
    # ----------------------------

    scale = [
        "LivingArea",
        "LotSizeSquareFeet",
        "AssociationFee",
        "DaysOnMarket"
    ]

    scale = [c for c in scale if c in train_df.columns]

    scaler = StandardScaler()
    train_df[scale] = scaler.fit_transform(train_df[scale])
    test_df[scale] = scaler.transform(test_df[scale])

    train_df = train_df.drop(columns=cat_col, errors='ignore')
    test_df = test_df.drop(columns=cat_col, errors='ignore')

    train_df = train_df.drop_duplicates()
    test_df = test_df.drop_duplicates()


    return train_df, test_df

In [15]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def mdape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.median(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
model_df['CloseDate'] = pd.to_datetime(model_df['CloseDate'])
model_df = model_df.sort_values('CloseDate').reset_index(drop=True)

model_results = []

# # Split dataframe into train and test sets using given window
def get_time_split(data, X_months):
    latest_date = data['CloseDate'].max()
    test_start_date = latest_date - pd.DateOffset(months=1)
    train_start_date = test_start_date - pd.DateOffset(months=X_months)
    test_set = data[data['CloseDate'] >= test_start_date]
    train_set = data[(data['CloseDate'] >= train_start_date) & (data['CloseDate'] < test_start_date)]
    return train_set, test_set



train, test = get_time_split(model_df, X_months=13)
train_df, test_df = preproc_df(train, test)

features = train_df.shape[1]
X_train = train_df.drop(columns=['ClosePrice', 'DaysOnMarket'])
y_train = train_df[target]
X_test = test_df.drop(columns=['ClosePrice', 'DaysOnMarket'])
y_test = test_df[target]

In [ ]:

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(max_depth=5),
    "Random Forest": RandomForestRegressor(n_estimators=100),
    "XGBoost": xgb.XGBRegressor(max_depth=8, learning_rate=0.10, n_estimators=700,
                                random_state=42, n_jobs=-1)
}

for model_name, model in models.items():

    model.fit(X_train, y_train)

    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)

    train_r2 = r2_score(y_train, train_preds)
    test_r2 = r2_score(y_test, test_preds)

    mae = mean_absolute_error(y_test, test_preds)
    rmse = np.sqrt(root_mean_squared_error(y_test,test_preds))

    model_results.append({
        "model": model_name,
        "train_r2": train_r2,
        "test_r2": test_r2,
        "mae": mae,
        "rmse": rmse,
        "mape": mape(y_test, test_preds),
        "mdape": mdape(y_test, test_preds)
    })

comparison = pd.DataFrame(model_results)

In [24]:
comparison.sort_values(by='test_r2', ascending=False)

,model,train_r2,test_r2,mae,rmse,mape,mdape
3,XGBoost,0.969049,0.916372,97572.893503,412.064202,10.284018,6.944414
2,Random Forest,0.987151,0.908278,100438.832467,421.691169,10.555295,6.926255
0,Linear Regression,0.828333,0.785506,169924.676311,521.471107,21.738001,13.369923
1,Decision Tree,0.659456,0.670616,227005.198731,580.500019,28.059530,19.888418


 - From comparing the models, XGBoost results in the highest R2 value followed by the tree models and finally linear regression.
 - The MdAPE values are smaller than the MAPE values indicating the data has a right-skewed distribution.

In [25]:
comparison.to_csv('metrics_summary.csv')